<a href="https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ritakimani9-lang/machinelearning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)
The ml task type i choose is classification which  predicts is_declining_label as down or not down in accordance to my Lane,"ranking signal analysis" with the research question (which 3-5 signals are worth attention). it does this through training where the model gives the signals with the highest scores. Ranking as an ML task type  is not feasible with my question as there are no set of items to be ordered - the signals do not need to be ordered.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**What would you predict? Where does that label come from — observed outcome or a defined rule?**

The target is **`is_declining_label`**, defined as:

`is_declining_label = trend_direction == "down"`

However, `trend_direction` is itself derived from `trend_pct`:

`trend_pct = (impressions_last_30d − impressions_prev_30d) / impressions_prev_30d × 100`

This means the target has two layers:

* **Layer 1 — Observed:** `trend_pct` is based on actual impressions measured across two completed time windows. The change in impressions is genuinely observed and counted, not predicted or guessed.
* **Layer 2 — Defined rule:** `trend_direction` converts this continuous percentage into categories using a human-defined ±20% threshold. The choice of 20% is a rule imposed on the data, not a naturally occurring boundary.

Therefore, **`is_declining_label` is a proxy rather than clean ground truth**. The underlying change in impressions is observed, but the decision that a decline beyond a particular threshold counts as "declining" is defined by us.

The proxy is intended to represent **meaningful loss of search visibility**. However, a page can experience a 25% drop in impressions for reasons unrelated to the quality of its content, such as seasonality, competitors gaining rankings, or a Google algorithm update. Therefore, **declining impressions do not necessarily mean that the content itself is bad or that the page is inherently declining in quality**.

### Leakage rule

Because `trend_pct` and `trend_direction` are used to construct `is_declining_label`, **they must not be used as model input features**. Otherwise, the model would be given information that directly defines the target, causing target leakage.

Finally, because `impressions_last_30d` and `impressions_prev_30d` are both already-completed periods at the time of export, this label describes a decline that **has already happened**. It therefore supports **diagnosing past declines**, not forecasting which pages will decline in the future.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Evaluation metric

**What single metric will you use to decide whether the model is good enough to trust downstream?**

The dataset contains **54.2% declining pages and 45.8% non-declining pages**. Therefore, a model that simply predicted "down" for every page would already achieve **54.2% accuracy**. This provides the baseline that any useful model must improve upon.

**Accuracy alone is therefore a weak choice.** For example, 58% accuracy might initially appear reasonable, but it is only slightly better than the 54.2% baseline. Accuracy also depends on the chosen classification threshold and does not show how well the model separates the two classes overall.

The primary evaluation metric will therefore be **ROC-AUC**. ROC-AUC measures how well the model ranks actual declining pages above non-declining pages across different classification thresholds. A value of **0.5 represents random performance**, while **1.0 represents perfect separation**. This makes ROC-AUC a useful measure of whether the model has learned meaningful signal rather than simply exploiting the class distribution.

The purpose of the classifier in this project is **not to build the best possible prediction system**. The classifier is a means to an end: its **feature importances** are the main downstream output. Therefore, ROC-AUC is being used to establish that the model has learned enough real separation between declining and non-declining pages for its feature importance results to be worth examining. If the AUC were close to 0.5, the model would have learned little useful signal, and its feature importances would not be trustworthy.

One limitation is that ROC-AUC only tells us how well the model separates the classes overall. It does not tell us which features are responsible for that separation, nor does it resolve the problem of correlated features potentially sharing or splitting importance.

**Defensible metric: ROC-AUC.**


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [9]:
import pandas as pd

github_csv_url = "https://raw.githubusercontent.com/ritakimani9-lang/machinelearning/refs/heads/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(github_csv_url)
display(df.head())

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
# Build is_declining_label
df['is_declining_label'] = df['trend_pct'] < -20

# Define columns to exclude due to leakage or being part of the target calculation
exclude_columns = [
    'trend_direction',
    'trend_pct',
    'content_id',
    'client_id',
    'provider_used',
    'model_used',
    'impressions_last_30d',
    'impressions_prev_30d'
]

# Get all columns from the original DataFrame
all_df_columns = df.columns.tolist()

# Filter out the excluded columns to get the candidate features
candidate_features = [col for col in all_df_columns if col not in exclude_columns and col != 'is_declining_label']

# Create the working DataFrame with only the candidate features and the new target label
df_working = df[candidate_features + ['is_declining_label']].copy()

# Display the first 5 rows of the working DataFrame
display(df_working.head())

,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,impressions_90d,clicks_90d,...,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,is_declining_label
0,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,3803,29,...,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,True
1,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,15320,7,...,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,True
2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,12581,11,...,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,True
3,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,11751,58,...,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,False
4,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,19140,24,...,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,True


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML rather than a simple rule?

**Could a simple if/else rule do this job, or does it need to be messier than that?**

A simple rule someone might initially propose is: **"older content + worse search position = likely declining."** This is a reasonable common-sense assumption, but the Week 1 data does not support it.

Among the observed pages, declining pages had an average position of **15.9**, compared with **16.8** for non-declining pages. They were also **younger on average**, at **236 days**, compared with **280 days** for non-declining pages. Therefore, both signals point in the opposite direction from the naive rule: declining pages were not older or worse-positioned on average.

This does not by itself prove that ML will perform better. It shows that these individual signals do not provide a straightforward rule for identifying declining pages. Their information may be weak on their own, or their meaning may depend on their **combination with other signals**. For example, the effect of content age might differ depending on search volume, freshness, or other page characteristics. Group averages alone cannot reveal these interactions.

With **17+ candidate signals**, manually testing every possible combination would quickly become impractical. There are already hundreds of possible pairwise combinations before considering three-way or higher-order interactions. A machine-learning model can evaluate these signals simultaneously and potentially identify patterns that are difficult to specify as hand-written if/else rules.

Therefore, the evidence does not justify claiming that ML is guaranteed to be better. It justifies **trying ML because the simple single-signal rule visibly fails on the observed data, while the problem may involve combinations of many signals**.


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.